# Optional project diagnostics

This notebook checks the runtime, VOC data, contracts and previews. It is not required for training. Use `train.ipynb` to train directly.

Attach the source dataset and use the same version as your teammate. The source root is the `source/` directory inside the Kaggle input mount. Restart the kernel after updating the dataset.

Install missing diagnostic dependencies if needed:

```ipython
%pip install "rich-click>=1.8,<2" "pytest>=8,<10" "pydantic-settings>=2,<3"
```


In [ ]:
import sys
from pathlib import Path

project_dir = Path("/kaggle/input/datasets/denysmykhailov/alseg-source/source")
sys.path.insert(0, str(project_dir))
sys.path.insert(0, str(project_dir / "tests"))
from alseg.config import load_config

config_path = project_dir / "configs" / "base.toml"
setup_script = project_dir / "script" / "setup_project.py"
tests_dir = project_dir / "tests"
project_version = load_config(project_dir / "pyproject.toml")["project"]["version"]

In [ ]:
%run "$setup_script" --config "$config_path" --inspect-data

In [ ]:
%run -m pytest -- -q -o cache_dir=/kaggle/working/alseg/pytest-cache "$tests_dir"

In [ ]:
from synthetic import make_fixture

from alseg.core.selection import validate_selection

for dimension in (2, 5):
    fixture = make_fixture(dimension)
    validate_selection(fixture.request, fixture.expected_top_uncertainty)
    print("Embedding shape:", tuple(fixture.request.embeddings.shape))
    print(
        "Expected top-uncertainty IDs (tie-break by ID):",
        fixture.expected_top_uncertainty,
    )

# Person 2 implements Selector.select(request) in adapters/acquisition.py.
# validate_selection checks output validity; algorithm tests check correctness.

In [ ]:
import importlib.metadata
import json
import platform

import torch

setup_dir = Path("/kaggle/working/alseg/setup")
setup_dir.mkdir(parents=True, exist_ok=True)
environment = {
    "project_version": project_version,
    "source_dir": str(project_dir),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu_names": [
        torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())
    ],
}
(setup_dir / "environment.json").write_text(json.dumps(environment, indent=2) + "\n")
packages = sorted(
    {
        f"{dist.metadata['Name']}=={dist.version}"
        for dist in importlib.metadata.distributions()
        if dist.metadata["Name"]
    }
)
(setup_dir / "requirements-kaggle.txt").write_text("\n".join(packages) + "\n")
print(json.dumps(environment, indent=2))
print("Environment snapshot:", setup_dir)

In [ ]:
import uuid

import numpy as np
from IPython.display import display
from PIL import Image

from alseg.adapters.voc import VocEvaluation, VocImages, VocOracle
from alseg.data.voc import find_voc_root, load_manifest

data_config = load_config(config_path)["data"]
explicit_root = data_config["voc_root"]
voc_root = (
    Path(explicit_root)
    if explicit_root
    else find_voc_root(Path(data_config["dataset_root"]))
)
manifest = load_manifest(voc_root)
pool = VocImages(voc_root, manifest)
evaluation = VocEvaluation(voc_root, manifest)
# This diagnostic query is a separate preview run, never part of benchmark L.
preview_dir = Path("/kaggle/working/alseg/previews") / uuid.uuid4().hex
oracle = VocOracle(voc_root, manifest, preview_dir / "oracle_queries.jsonl")
image = pool.get_image(0)
mask = oracle.query([0])[0]
colors = np.array(
    [(i * 37 % 256, i * 67 % 256, i * 97 % 256) for i in range(256)], dtype=np.uint8
)
colors[255] = [255, 0, 255]  # Void, ignored during training/evaluation.
rgb = Image.fromarray(image.permute(1, 2, 0).numpy())
colored_mask = Image.fromarray(colors[mask.numpy()])
print("Train ID 0:", manifest.pool_ids[0], "classes:", mask.unique().tolist())
print("RGB | class mask (void=magenta) | overlay")
preview = Image.new("RGB", (rgb.width * 3, rgb.height))
for index, panel in enumerate((rgb, colored_mask, Image.blend(rgb, colored_mask, 0.5))):
    preview.paste(panel, (index * rgb.width, 0))
display(preview)
print(
    "Val ID 0:",
    manifest.val_ids[0],
    "classes:",
    evaluation.get_mask(0).unique().tolist(),
)
print("Preview query audit:", preview_dir / "oracle_queries.jsonl")

## Next steps

Implement acquisition adapters against `alseg/interfaces/`. Complete cumulative rounds and held-out calibration before the benchmark. The head training entry point below already supports W&B and final official-val metrics.


## Frozen encoder smoke run (GPU)

After attaching the source dataset release with the encoder implementation, run these commands in a code cell. They download the pinned checkpoint if needed and write only to `/kaggle/working/`. Omit `--limit 4` for full-pool embeddings. See README for preprocessing and VOC pretraining overlap.

```ipython
feature_script = project_dir / "script" / "extract_features.py"
%run "$feature_script" --config "$config_path" --device cuda --limit 4
```


## Segmentation head and W&B

Install the missing dependencies without replacing Kaggle's PyTorch:

```ipython
%pip install "pydantic-settings>=2,<3" "lightning==2.6.6" "wandb==0.28.1"
```

Enable a Kaggle Add-ons > Secrets entry named **WANDB_API_KEY**. Online tracking
reads it automatically and uses entity **public-team**. Do not put the key in a cell.
This trains one Random budget for the configured 20 epochs and reports all official
val images. It does not yet perform cumulative AL rounds or held-out calibration.

```ipython
train_script = project_dir / "script" / "train_segmentation.py"
%run "$train_script" --config "$config_path" --budget 21 --device cuda
```

Use `--wandb-mode offline` to keep W&B data local or `disabled` for local artifacts
only. No secret is requested in either mode. See README for metrics and artifacts.
